# CNN pour la classification d'Alzheimer

Ce projet utilise un **réseau de neurones convolutif (CNN)** avec TensorFlow/Keras pour classifier des images IRM du cerveau en 4 catégories : MildDemented, ModerateDemented, NonDemented et VeryMildDemented.

## Chargement des données
La fonction `load_images_and_labels()` parcourt les dossiers du dataset, charge les images, les redimensionne en **150x150**, les convertit en tableaux numpy et normalise les pixels entre **0 et 1**. Chaque image reçoit un label correspondant à sa catégorie.

## Division du dataset
Les données sont divisées en **80% pour l'entraînement** et **20% pour le test** avec `train_test_split`.

## Construction du modèle
La fonction `build_cnn()` crée un modèle **Sequential** composé de :
- couches **Conv2D** pour extraire les caractéristiques
- couches **MaxPooling** pour réduire la dimension
- **Flatten** pour transformer les features en vecteur
- **Dense + Softmax** pour classifier les images.

Le modèle est compilé avec l'optimiseur **Adam** et la fonction de perte **sparse_categorical_crossentropy**.

## Entraînement et évaluation
Le modèle est entraîné pendant **20 epochs** puis évalué sur les données de test. Les métriques calculées sont **accuracy, precision, recall et F1-score**.

## Sauvegarde du modèle
Le modèle final est sauvegardé dans le dossier **Saved_Models** sous le nom `mon_modele_cnn.keras`.

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score
from PIL import UnidentifiedImageError

# Chemin vers le dossier contenant les images
data_directory = "Alzheimer_s Dataset"
categories = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
image_size = (150, 150)

# Fonction pour charger les images avec gestion des erreurs
def load_images_and_labels(data_directory, categories, image_size):
    images = []
    labels = []
    valid_extensions = ('.jpg', '.jpeg', '.png')
    for category in categories:
        category_index = categories.index(category)
        category_path = os.path.join(data_directory, category)
        for filename in os.listdir(category_path):
            img_path = os.path.join(category_path, filename)
            if os.path.isdir(img_path) or not filename.lower().endswith(valid_extensions):
                continue
            try:
                img = load_img(img_path, target_size=image_size)
                img_array = img_to_array(img)
                images.append(img_array)
                labels.append(category_index)
            except UnidentifiedImageError:
                print(f"[Ignoré] Fichier non reconnu comme image : {img_path}")
            except Exception as e:
                print(f"[Erreur] {img_path} : {str(e)}")

    images = np.array(images, dtype='float32') / 255.0
    labels = np.array(labels)
    return images, labels

# Chargement des images
images, labels = load_images_and_labels(data_directory, categories, image_size)
train_images, test_images, train_labels, test_labels = train_test_split(images, labels, test_size=0.2, random_state=42)

# Définition du modèle CNN
def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Entraînement du modèle
model = build_cnn((150, 150, 3), len(categories))
model.fit(train_images, train_labels, epochs=20, validation_data=(test_images, test_labels))

# Évaluation du modèle
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f'Test accuracy: {test_accuracy:.2f}, Test loss: {test_loss:.2f}')

# Prédiction et métriques
y_pred = np.argmax(model.predict(test_images), axis=-1)
precision = precision_score(test_labels, y_pred, average='weighted')
recall = recall_score(test_labels, y_pred, average='weighted')
f1 = f1_score(test_labels, y_pred, average='weighted')
accuracy = accuracy_score(test_labels, y_pred)

# Affichage des résultats
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print(f'Accuracy: {accuracy:.2f}')
print(classification_report(test_labels, y_pred, target_names=categories))

# Sauvegarde du modèle
save_path = 'Saved_Models'
os.makedirs(save_path, exist_ok=True)
model.save(os.path.join(save_path, 'mon_modele_cnn.keras'))
print(f"Modèle sauvegardé sous '{save_path}/mon_modele_cnn.keras'.")


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 48s 299ms/step - accuracy: 0.5029 - loss: 1.0167 - val_accuracy: 0.5791 - val_loss: 0.8578
Epoch 2/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 49s 313ms/step - accuracy: 0.6330 - loss: 0.7939 - val_accuracy: 0.6900 - val_loss: 0.7023
Epoch 3/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 47s 302ms/step - accuracy: 0.7941 - loss: 0.4983 - val_accuracy: 0.8474 - val_loss: 0.3764
Epoch 4/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 48s 307ms/step - accuracy: 0.9267 - loss: 0.2188 - val_accuracy: 0.9357 - val_loss: 0.1825
Epoch 5/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 46s 296ms/step - accuracy: 0.9749 - loss: 0.0746 - val_accuracy: 0.9598 - val_loss: 0.1368
Epoch 6/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 47s 304ms/step - accuracy: 0.9943 - loss: 0.0261 - val_accuracy: 0.9068 - val_loss: 0.2513
Epoch 7/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 54s 348ms/step - accuracy: 0.9806 - loss: 0.0523 - val_accuracy: 0.9775 - val_loss: 0.0811
Epoch 8/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 53s 340ms/step - accuracy: 1.0000 - loss: 0